Poseidon MDS matrix and constants from 

In [1]:
from sage.all import *
from Crypto.Hash import SHAKE256

In [2]:
def int_to_4_u64(x: int):
    mask = (1 << 64) - 1
    return [
        (x >> (64 * 0)) & mask,
        (x >> (64 * 1)) & mask,
        (x >> (64 * 2)) & mask,
        (x >> (64 * 3)) & mask,
    ]

In [3]:
def u_64_chunks_to_int(chunks):
    x = 0
    for el in reversed(chunks):
        x *= 2**64
        x += el
    return x

In [4]:
def u8_to_u64(u8_list):
    assert len(u8_list) == 32
    
    result = []
    
    for i in range(0, 32, 8):
        value = 0
        for j in range(8):
            value |= u8_list[i + j] << (8 * j)  # little endian
        result.append(value)
        
    return result

In [5]:
q = 0x73eda753299d7d483339d80809a1d80553bda402fffe5bfeffffffff00000001

n = 4

rounds = 23

d = 5

q_1_2 = (q - 1) >> 1

int_to_4_u64(q_1_2)

[9223372034707292160,
 12240451741123816959,
 1845609449319885826,
 4176758429732224676]

In [6]:
gcd(d + q_1_2, q - 1)

1

In [7]:
n = 4
mds_4 = [
        [
            0x73eda753299d7d483339d80809a1d80553bda402fffe5bfefffffffefffe3470,
            0x73eda753299d7d483339d80809a1d80553bda402fffe5bfefffffffefd31ed71,
            0x73eda753299d7d483339d80809a1d80553bda402fffe5bfefffffffb29e8dccf,
            0x73eda753299d7d483339d80809a1d80553bda402fffe5bfefffffad750e8b4d1,
        ],
        [
            0x00000000000000000000000000000000000000000000000000000000000217f0,
            0x0000000000000000000000000000000000000000000000000000000003439b6f,
            0x00000000000000000000000000000000000000000000000000000004767d6c50,
            0x000000000000000000000000000000000000000000000000000005ff275b59ce,
        ],
        [
            0x73eda753299d7d483339d80809a1d80553bda402fffe5bfefffffffeffffb213,
            0x73eda753299d7d483339d80809a1d80553bda402fffe5bfefffffffeff885411,
            0x73eda753299d7d483339d80809a1d80553bda402fffe5bfefffffffe5cba96b4,
            0x73eda753299d7d483339d80809a1d80553bda402fffe5bfeffffff23ae5f0fb1,
        ],
        [
            0x0000000000000000000000000000000000000000000000000000000000000190,
            0x0000000000000000000000000000000000000000000000000000000000022312,
            0x0000000000000000000000000000000000000000000000000000000002df2030,
            0x00000000000000000000000000000000000000000000000000000003d95ce1b3,
        ],
]

In [8]:
for i in range(0, n):
    for j in range(0, n):
        mds_4[i][j] = int_to_4_u64(mds_4[i][j])

In [9]:
print("[")
for i in range(0, n):
    print("[")
    for j in range(0, n):
        print(str(mds_4[i][j]) + ",")
    print("],")
print("]")

[
[
[18446744069414466672, 6034159408538082302, 3691218898639771653, 8353516859464449352],
[18446744069367524721, 6034159408538082302, 3691218898639771653, 8353516859464449352],
[18446744052937841871, 6034159408538082302, 3691218898639771653, 8353516859464449352],
[18446738401415181521, 6034159408538082302, 3691218898639771653, 8353516859464449352],
],
[
[137200, 0, 0, 0],
[54762351, 0, 0, 0],
[19167800400, 0, 0, 0],
[6593435097550, 0, 0, 0],
],
[
[18446744069414564371, 6034159408538082302, 3691218898639771653, 8353516859464449352],
[18446744069406741521, 6034159408538082302, 3691218898639771653, 8353516859464449352],
[18446744066675349172, 6034159408538082302, 3691218898639771653, 8353516859464449352],
[18446743127447244721, 6034159408538082302, 3691218898639771653, 8353516859464449352],
],
[
[400, 0, 0, 0],
[140050, 0, 0, 0],
[48177200, 0, 0, 0],
[16531644851, 0, 0, 0],
],
]


In [10]:
seed = ("Grendel_" + str(q) + "_n_" + str(n) + "_rounds_" + str(rounds)).encode('ascii')
shake = SHAKE256.new()
shake.update(seed)
byte_length = rounds * n * ceil(log(q, 2**8))
byte_length

2944

In [11]:
constants_raw = list(shake.read(byte_length))
constants = []

for r in range(0, rounds):
    cnst = []
    for i in range(0, n):
        c = u_64_chunks_to_int(u8_to_u64(constants_raw[(n * r + i) * ceil(log(q, 2**8)):(n * r + i + 1) * ceil(log(q, 2**8))]))
        c = c % q
        cnst.append(int_to_4_u64(c))
    constants.append(cnst)

In [12]:
print("[")
for cnst in constants:
    txt = "["
    for c in cnst:
        txt += str(c) + ","
    txt += "],"
    print(txt)
print("]")

[
[[8996368599066996084, 7161782218847590339, 13569724898000490312, 4886535599062989821],[13513274013872582046, 13319035137997813708, 7621869252739544927, 6628455542068877147],[6787925051431542439, 551188582126291554, 6424749108514621415, 1280800187394125528],[11259236945602777691, 15717626683679769612, 7703323096413752049, 7370910369883044513],],
[[4819153468715786006, 14312958908043369356, 16184198421156776217, 2785117492359158813],[5828097840602114351, 4952126853554472894, 15530309300194245211, 4743658413438861253],[10568888962510415065, 1583007090622436116, 2033560798719956693, 1396127043997968934],[4647857757358483298, 3888318114514708208, 11338811471293572960, 197493640451918327],],
[[14428983644279436727, 15222431796101347257, 13814391239704522491, 6271118214323726689],[15984605592654072737, 14556002948387729960, 10959127605552307647, 966178953063508056],[5286992370954542631, 3563850546964710736, 15484039933145838607, 7287286420692236748],[17835518760023891346, 30603104278894641